In [1]:
import json
import joblib
import pandas as pd
import wandb
from dotenv import load_dotenv
from tune import run_study
from train import run_cv, train_final, RANDOM_SEED

load_dotenv()

df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

/Users/aleksandragakin/projects/study/spaceship_titanic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tune

In [2]:
N_TRIALS = 100
CV_FOLDS = 10
STUDY_NAME = "catboost-v2"

study = run_study(df, n_trials=N_TRIALS, study_name=STUDY_NAME, cv_folds=CV_FOLDS)

  0%|          | 0/100 [00:00<?, ?it/s]wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
Best trial: 82. Best value: 0.81974: 100%|██████████| 100/100 [5:31:31<00:00, 198.92s/it]  


Best CV:     0.81974
Best params: {'iterations': 3412, 'learning_rate': 0.006446831836813848, 'depth': 7, 'l2_leaf_reg': 4.4217638410210895, 'bagging_temperature': 0.8818018101838497}
Saved best_params.json


## Train

In [3]:
with open("best_params.json") as f:
    params = json.load(f)

mean, std = run_cv(params, df, cv_folds=CV_FOLDS)
print(f"CV: {mean:.5f} ± {std:.5f}")

CV: 0.81974 ± 0.01038


In [4]:
from features import PIPELINE_STEPS

pipeline = train_final(df, params)

wandb.init(project="spaceship-titanic", job_type="train", config={
    **params,
    "model": "catboost",
    "cv_folds": CV_FOLDS,
    "pipeline_steps": PIPELINE_STEPS,
})
wandb.log({"cv_mean": mean, "cv_std": std})
wandb.finish()

0:	learn: 0.6900011	total: 4.16ms	remaining: 14.2s
100:	learn: 0.5146481	total: 422ms	remaining: 13.8s
200:	learn: 0.4548393	total: 848ms	remaining: 13.6s
300:	learn: 0.4267588	total: 1.28s	remaining: 13.3s
400:	learn: 0.4089928	total: 1.72s	remaining: 12.9s
500:	learn: 0.3966150	total: 2.17s	remaining: 12.6s
600:	learn: 0.3877031	total: 2.63s	remaining: 12.3s
700:	learn: 0.3812235	total: 3.1s	remaining: 12s
800:	learn: 0.3751397	total: 3.59s	remaining: 11.7s
900:	learn: 0.3701996	total: 4.08s	remaining: 11.4s
1000:	learn: 0.3658934	total: 4.57s	remaining: 11s
1100:	learn: 0.3613410	total: 5.07s	remaining: 10.6s
1200:	learn: 0.3574998	total: 5.55s	remaining: 10.2s
1300:	learn: 0.3536448	total: 6.04s	remaining: 9.79s
1400:	learn: 0.3499653	total: 6.52s	remaining: 9.36s
1500:	learn: 0.3460221	total: 7.02s	remaining: 8.93s
1600:	learn: 0.3421320	total: 7.52s	remaining: 8.51s
1700:	learn: 0.3378919	total: 8.03s	remaining: 8.08s
1800:	learn: 0.3337736	total: 8.96s	remaining: 8.01s
1900:	lea

wandb: Currently logged in as: agakinalexandr (agakinalexandr-singidunum-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


cv_mean,▁
cv_std,▁
cv_mean,0.81974
cv_std,0.01038


## Predict

In [5]:
pipeline = joblib.load("pipeline.pkl")

passenger_ids = test_df["PassengerId"].copy()
preds = pipeline.predict(test_df).astype(bool)

submission = pd.DataFrame({"PassengerId": passenger_ids, "Transported": preds})
submission.to_csv("submission.csv", index=False)

n_transported = preds.sum()
wandb.init(
    project="spaceship-titanic",
    job_type="predict",
    config=pipeline.named_steps["model"].get_params(),
)
wandb.log({
    "n_rows": len(submission),
    "n_transported": int(n_transported),
    "transport_rate": round(n_transported / len(submission), 4),
})
wandb.finish()

print(f"Saved submission.csv ({len(submission)} rows)")
print(f"Transported: {n_transported} / {len(submission)} ({n_transported / len(submission):.1%})")

n_rows,▁
n_transported,▁
transport_rate,▁
n_rows,4277
n_transported,2137
transport_rate,0.4996


Saved submission.csv (4277 rows)
Transported: 2137 / 4277 (50.0%)
